In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. 加载数据
train = pd.read_csv('train.csv')  # 训练数据
test = pd.read_csv('test.csv')    # 测试数据

In [2]:
train['days'] = pd.to_datetime(train['date'])
test['days'] = pd.to_datetime(test['date'])
base_date = pd.to_datetime('2010-01-01')
train['days'] = (train['days'] - base_date).dt.days
test['days'] = (test['days'] - base_date).dt.days
train['days'].head()

0    0
1    0
2    0
3    0
4    0
Name: days, dtype: int64

In [3]:
train = pd.get_dummies(train, columns=['country'], drop_first=True)  # 独热编码
test = pd.get_dummies(test, columns=['country'], drop_first=True)
train = pd.get_dummies(train, columns=['store'], drop_first=True)  # 独热编码
test = pd.get_dummies(test, columns=['store'], drop_first=True)

In [4]:
train['num_sold'] = train['num_sold'].fillna(0)

In [5]:
features = ['days', 'country_Finland', 'country_Italy', 'country_Norway', 'country_Kenya', 'country_Singapore', 'store_Premium Sticker Mart', 'store_Stickers for Less']
X = train[features]
Y = train['num_sold']
X_test = test[features]

X_scaled = StandardScaler().fit_transform(X)
X_test_scaled = StandardScaler().fit_transform(X_test)


In [6]:
X_train, X_val, Y_train, Y_val = train_test_split(X_scaled, Y, test_size=0.2, random_state=42)
svm = SVC(kernel='linear', C=1, gamma=0.1, random_state=42)

# 模型训练
svm.fit(X_train, Y_train)

SVC(C=1, gamma=0.1, kernel='linear', random_state=42)

In [ ]:
from tqdm import tqdm
from sklearn.metrics import mean_squared_error
from sklearn.svm import SVR  # 也可换成你的模型

# 1. 假设你已经完成模型训练，如：
# model = SVR().fit(X_train, y_train)

# 2. 验证集特征和标签
#    这里以 X_val, y_val 为例

batch_size = 1000  # 每批预测多少条，可按需调整
n_samples = X_val.shape[0]

all_preds = []  # 用于保存每个批次的预测结果

# 用 tqdm 显示分批预测进度
for start_idx in tqdm(range(0, n_samples, batch_size), desc="Predicting in batches"):
    end_idx = min(start_idx + batch_size, n_samples)
    
    # 取本批次的数据和标签
    X_batch = X_val[start_idx:end_idx]
    y_batch_true = Y_val[start_idx:end_idx]
    
    # 调用模型对本批次进行预测
    y_batch_pred = svm.predict(X_batch)
    
    # 保存预测结果
    all_preds.append(y_batch_pred)
    
    # 计算本批次的 MSE
    batch_mse = mean_squared_error(y_batch_true, y_batch_pred)
    
    # 用 tqdm.write() 打印当前批次的 MSE，避免与进度条冲突
    tqdm.write(f"Batch {start_idx // batch_size} MSE: {batch_mse:.4f}")

# 合并所有批次的预测结果
all_preds = np.concatenate(all_preds, axis=0)

# 计算整体验证集上的 MSE
final_mse = mean_squared_error(Y_val, all_preds)
print(f"\nOverall Validation MSE: {final_mse:.4f}")


Predicting in batches:   2%|▏         | 1/47 [15:32<11:54:37, 932.12s/it]

Batch 0 MSE: 664734.7380
